# Descriptives

Corpus, accusations, metadata coverage, and the per-country accusation rate.
Country exclusions are applied throughout, so these numbers describe the
**analysis corpus** rather than the raw pipeline output.

Produces two appendix outputs:

- `outputs/desc_country_rates.png` — accusation rate by country
- `outputs/country_rates.tex` — the same as a LaTeX table

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys; sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from lib import data, viz
from lib.codebooks import EXCLUDED_COUNTRIES, TARGET_TYPE
viz.apply_style()

con = data.duck()
print(f"excluded: {sorted(EXCLUDED_COUNTRIES)}")

## 1. Corpus

In [ ]:
corpus = con.execute("""
    SELECT COUNT(*)                         AS sentences,
           COUNT(DISTINCT source_speech_id) AS speeches,
           COUNT(DISTINCT speaker)          AS speakers,
           COUNT(DISTINCT country)          AS countries,
           COUNT(DISTINCT source_dataset)   AS datasets,
           MIN(date) AS first_date, MAX(date) AS last_date
    FROM corpus
""").df().T
corpus.columns = [""]
print(corpus.to_string())

## 2. Accusations

In [ ]:
acc = con.execute("""
    SELECT COUNT(*)                                             AS accusations,
           COUNT(DISTINCT accuser_speaker_id)                   AS distinct_accusers,
           SUM(CASE WHEN is_interjection = 1 THEN 1 ELSE 0 END) AS interjections
    FROM accusations
""").df().T
acc.columns = [""]
print(acc.to_string())

n_acc = con.execute("SELECT COUNT(*) FROM accusations").fetchone()[0]
n_sen = con.execute("SELECT COUNT(*) FROM corpus").fetchone()[0]
print(f"\nbase rate: {n_acc / n_sen * 100:.3f}% of sentences "
      f"({n_acc / n_sen * 10_000:.1f} per 10,000)")

In [ ]:
tt = con.execute("""
    SELECT target_type, COUNT(*) AS n
    FROM accusations GROUP BY 1 ORDER BY 2 DESC
""").df()
tt["label"] = tt["target_type"].map(TARGET_TYPE).fillna(tt["target_type"])
tt["%"] = (tt["n"] / tt["n"].sum() * 100).round(1)
print("who is accused:\n")
print(tt[["label", "n", "%"]].to_string(index=False))

## 3. Metadata coverage

Per distinct speaker in the corpus, not per accusation.

In [ ]:
sp = con.execute("""
    SELECT DISTINCT speaker_speaker_id     AS sid,
           speaker_gender                  AS gender,
           speaker_birth_year              AS birth_year,
           speaker_highest_isced           AS isced,
           speaker_career_sectors          AS sectors,
           speaker_partyfacts_id           AS pf,
           speaker_left_right              AS lr,
           speaker_populism                AS pop,
           speaker_in_cabinet              AS cab
    FROM corpus WHERE speaker_speaker_id IS NOT NULL
""").df()

n_sp = sp["sid"].nunique()
rows = []
for label, col in [("gender", "gender"), ("birth year", "birth_year"),
                   ("education (ISCED)", "isced"), ("career sectors", "sectors"),
                   ("party (PartyFacts)", "pf"),
                   ("ideology (V-Party)", "lr"), ("populism (V-Party)", "pop"),
                   ("incumbency (ParlGov)", "cab")]:
    k = int(sp[col].notna().sum())
    rows.append((label, f"{k:,}", f"{k / max(n_sp, 1) * 100:.1f}"))

print(f"speakers in the analysis corpus: {n_sp:,}\n")
print(pd.DataFrame(rows, columns=["variable", "n", "% of speakers"])
        .to_string(index=False))

print(f"\ndistinct parties: {int(sp['pf'].nunique()):,}")

## 4. Accusation rate by country

The appendix figure and table. The rate is accusations per 10,000 sentences, so
countries with larger corpora are not advantaged.

In [ ]:
byc = con.execute("""
    WITH s AS (
        SELECT country,
               COUNT(*) AS sentences,
               COUNT(DISTINCT source_speech_id) AS speeches,
               COUNT(DISTINCT speaker) AS speakers,
               MIN(CAST(substr(date, 1, 4) AS INT)) AS first_year,
               MAX(CAST(substr(date, 1, 4) AS INT)) AS last_year
        FROM corpus WHERE date IS NOT NULL AND length(date) >= 4
        GROUP BY 1
    ), a AS (
        SELECT country, COUNT(*) AS accusations FROM accusations GROUP BY 1
    )
    SELECT s.*, COALESCE(a.accusations, 0) AS accusations
    FROM s LEFT JOIN a USING (country)
""").df()

byc["rate_10k"] = byc["accusations"] / byc["sentences"] * 10_000
byc = byc.sort_values("rate_10k", ascending=False).reset_index(drop=True)

pooled = byc["accusations"].sum() / byc["sentences"].sum() * 10_000
print(f"pooled rate: {pooled:.1f} per 10,000 sentences\n")
print(byc[["country", "first_year", "last_year", "sentences", "speeches",
           "speakers", "accusations", "rate_10k"]]
        .round(1).to_string(index=False))

In [ ]:
d = byc.sort_values("rate_10k")

fig, ax = plt.subplots(figsize=(8, 0.33 * len(d) + 1.5))
ax.barh(np.arange(len(d)), d["rate_10k"], color="#2f6f9f", height=0.7)
ax.axvline(pooled, color="#c0603f", lw=1.5, ls="--",
           label=f"pooled rate ({pooled:.1f})")
ax.set_yticks(np.arange(len(d)))
ax.set_yticklabels(d["country"])
ax.set_xlabel("accusations per 10,000 sentences")
ax.set_title("Accusation rate by country")
ax.legend(loc="lower right")
for i, (r, n) in enumerate(zip(d["rate_10k"], d["accusations"])):
    ax.text(r + 0.6, i, f"{n:,}", va="center", fontsize=7, color="#555555")
fig.tight_layout()
viz.savefig(fig, "desc_country_rates")
plt.show()

### LaTeX table for the appendix

In [ ]:
rows = []
for r in byc.itertuples(index=False):
    rows.append(f"{r.country} & {r.first_year}--{r.last_year} & "
                f"{r.sentences:,} & {r.accusations:,} & {r.rate_10k:.1f} \\\\")

body = "\n".join(rows)
tex = f"""\\begin{{table}}[htbp]
\\centering
\\small
\\caption{{Accusation rate by country. The rate is accusations per 10,000
sentences. Countries excluded on data-quality grounds are not shown.}}
\\label{{tab:country-rates}}
\\begin{{tabular}}{{llrrr}}
\\toprule
Country & Period & Sentences & Accusations & Per 10{{,}}000 \\\\
\\midrule
{body}
\\midrule
Total & & {byc['sentences'].sum():,} & {byc['accusations'].sum():,} & {pooled:.1f} \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}
"""

out = "../outputs/country_rates.tex"
with open(out, "w") as f:
    f.write(tex)
print(f"wrote {out}\n")
print(tex)

## 5. Rate by source

Different corpora are transcribed and translated differently. Large gaps between
sources for the same country would be a measurement warning.

In [ ]:
byd = con.execute("""
    WITH s AS (
        SELECT source_dataset, country, COUNT(*) AS sentences
        FROM corpus GROUP BY 1, 2
    ), a AS (
        SELECT source_dataset, country, COUNT(*) AS accusations
        FROM accusations GROUP BY 1, 2
    )
    SELECT s.source_dataset, s.country, s.sentences,
           COALESCE(a.accusations, 0) AS accusations
    FROM s LEFT JOIN a USING (source_dataset, country)
""").df()
byd["rate_10k"] = (byd["accusations"] / byd["sentences"] * 10_000).round(1)
print(byd.sort_values("sentences", ascending=False).to_string(index=False))

multi = byd.groupby("country").filter(lambda g: len(g) > 1)
if len(multi):
    print("\ncountries covered by more than one source:")
    print(multi.sort_values(["country", "source_dataset"]).to_string(index=False))